# 🧠 Étape 3 : Sélection et Évaluation des Modèles
**Objectif :** Mettre en compétition 3 algorithmes de Machine Learning pour identifier le plus performant et le plus adapté à notre cas d'usage de détection d'anomalies (Smurfs).
Modèles testés :
1. **Régression Logistique :** Modèle linéaire, très explicable (White-box).
2. **Random Forest :** Modèle d'ensemble basé sur des arbres, gère bien la non-linéarité.
3. **Gradient Boosting :** Modèle d'ensemble séquentiel, redoutable sur les données tabulaires.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1. Chargement et Split Stratifié
df = pd.read_csv('../data/rocket_league_skill_data.csv')
X = df.drop(columns=['Target'])
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Définition des Pipelines (StandardScaler + Modèle)
models = {
    "Régression Logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest": make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(StandardScaler(), GradientBoostingClassifier(n_estimators=100, random_state=42))
}

### 🏆 Benchmark par Validation Croisée (Cross-Validation)
Nous évaluons les modèles sur 5 plis (folds) pour garantir la robustesse des résultats. Nous observons l'Accuracy, mais surtout le **Recall** (capacité à attraper les smurfs) et la **Precision** (ne pas accuser un innocent).

In [3]:
results = []

for name, pipeline in models.items():
    # On calcule plusieurs métriques en même temps
    cv_scores = cross_validate(
        pipeline, X_train, y_train, cv=5, 
        scoring=('accuracy', 'precision', 'recall'),
        n_jobs=-1
    )
    
    results.append({
        "Modèle": name,
        "Accuracy Moyenne": cv_scores['test_accuracy'].mean(),
        "Precision Moyenne": cv_scores['test_precision'].mean(),
        "Recall Moyen": cv_scores['test_recall'].mean()
    })

benchmark_df = pd.DataFrame(results).sort_values(by="Accuracy Moyenne", ascending=False)
display(benchmark_df)

,Modèle,Accuracy Moyenne,Precision Moyenne,Recall Moyen
0,Régression Logistique,0.823200,0.811517,0.842361
2,Gradient Boosting,0.815908,0.805196,0.834028
1,Random Forest,0.805836,0.791193,0.831250


**Conclusion de la Modélisation :**
La Régression Logistique surpasse les modèles ensemblistes (Random Forest, Gradient Boosting) sur toutes les métriques (Accuracy, Recall, Precision). Cela confirme que la relation entre les mécaniques de jeu et le rang est majoritairement linéaire. Ce résultat renforce d'autant plus notre choix pour la production : nous avons non seulement le modèle le plus performant, mais aussi le plus explicable (White-box), ce qui est indispensable pour justifier un bannissement.